# Cyclistic PySpark Analysis

Full-dataset analysis using the PySpark DataFrame API.

- **Data:** `data/processed/trips_clean.csv` (~14.8 million rows)
- **API:** DataFrame only — `SparkSession`, `df.filter`, `df.groupBy`, `df.select`, `df.show`
- **Timing:** each cell prints its own elapsed time; a summary cell at the end shows total run time

> `data/processed/trips_clean.csv` does not include `rideable_type` or lat/lng columns,
> so the bike-type section is omitted.

In [1]:
%pip install pyspark==3.5.1 python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp,
    hour, date_format,
    count, avg, stddev, min, max,
    percentile_approx, when,
    round as spark_round,
)

# Notebook-level timer — starts here, reported in the final cell
_NB_START = time.time()
print("Notebook timer started.")

Notebook timer started.


## Start Spark

In [3]:
_t = time.time()

spark = (
    SparkSession.builder
    .appName("CyclisticAnalysis")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

print(f"\nCell time: {time.time() - _t:.2f}s")

Spark version: 3.5.1

Cell time: 4.73s


## Load data

Read the full `trips_clean.csv` (~14.8 million rows).

In [4]:
_t = time.time()

DATA_PATH = "data/processed/trips_clean.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(DATA_PATH)
)

print("Columns:", df_raw.columns)
df_raw.show(5, truncate=False)

print(f"\nCell time: {time.time() - _t:.2f}s")

Columns: ['ride_id', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'member_casual']
+----------------+-------------------+-------------------+----------------------------+----------------+---------------------------+--------------+-------------+
|ride_id         |started_at         |ended_at           |start_station_name          |start_station_id|end_station_name           |end_station_id|member_casual|
+----------------+-------------------+-------------------+----------------------------+----------------+---------------------------+--------------+-------------+
|A847FADBBC638E45|2020-04-26 17:45:14|2020-04-26 18:12:03|Eckhart Park                |86              |Lincoln Ave & Diversey Pkwy|152.0         |member       |
|5405B80E996FF60D|2020-04-17 17:08:54|2020-04-17 17:17:03|Drake Ave & Fullerton Ave   |503             |Kosciuszko Park            |499.0         |member       |
|5DD24A79A4E006F4|2020-04-01 17:54:13|2020-04-

## Calculate ride duration

In [5]:
_t = time.time()

df = (
    df_raw
    .withColumn("started_at", to_timestamp("started_at", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ended_at",   to_timestamp("ended_at",   "yyyy-MM-dd HH:mm:ss"))
    .withColumn(
        "ride_length_minutes",
        (unix_timestamp("ended_at") - unix_timestamp("started_at")) / 60.0,
    )
)

df.select("ride_id", "started_at", "ended_at", "ride_length_minutes").show(5, truncate=False)

print(f"\nCell time: {time.time() - _t:.2f}s")

+----------------+-------------------+-------------------+-------------------+
|ride_id         |started_at         |ended_at           |ride_length_minutes|
+----------------+-------------------+-------------------+-------------------+
|A847FADBBC638E45|2020-04-26 17:45:14|2020-04-26 18:12:03|26.816666666666666 |
|5405B80E996FF60D|2020-04-17 17:08:54|2020-04-17 17:17:03|8.15               |
|5DD24A79A4E006F4|2020-04-01 17:54:13|2020-04-01 18:08:36|14.383333333333333 |
|2A59BBDF5CDBA725|2020-04-07 12:50:19|2020-04-07 13:02:31|12.2               |
|27AD306C119C6158|2020-04-18 10:22:59|2020-04-18 11:15:54|52.916666666666664 |
+----------------+-------------------+-------------------+-------------------+
only showing top 5 rows


Cell time: 0.31s


## Quality checks

Count missing values, summarise ride length, then filter out rides that are
zero/negative or longer than 24 hours.

In [6]:
_t = time.time()

# Missing values per column
print("Missing values per column:")
df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
).show()

# Ride length summary
print("Ride length (minutes) summary:")
df.select("ride_length_minutes").describe().show()

# Filter
MAX_DURATION = 24 * 60  # 1 440 minutes

total_before = df.count()
print("Number of rides before filtering:", total_before)

df_clean = (
    df.filter(
        (col("ride_length_minutes") > 0)
        & (col("ride_length_minutes") <= MAX_DURATION)
    )
    .cache()
)

total_after = df_clean.count()   # triggers cache
print("Number of rides after filtering:",  total_after)

print(f"\nCell time: {time.time() - _t:.2f}s")

Missing values per column:
+-------+----------+--------+------------------+----------------+----------------+--------------+-------------+-------------------+
|ride_id|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|member_casual|ride_length_minutes|
+-------+----------+--------+------------------+----------------+----------------+--------------+-------------+-------------------+
|      0|         0|       0|           1618529|         1619152|         1742793|       1743254|            0|                  0|
+-------+----------+--------+------------------+----------------+----------------+--------------+-------------+-------------------+

Ride length (minutes) summary:
+-------+-------------------+
|summary|ride_length_minutes|
+-------+-------------------+
|  count|           14804463|
|   mean| 21.675852048804746|
| stddev|  265.1126543321353|
|    min|-29049.966666666667|
|    max|           156390.4|
+-------+-------------------+

Number of 

## Member vs casual

Compare ride lengths and trip counts between members and casual riders.

In [8]:
_t = time.time()

# Ride length stats by user type
print("Ride length (minutes) by user type:")
df_clean.groupBy("member_casual").agg(
    count("ride_length_minutes").alias("count"),
    spark_round(avg("ride_length_minutes"),    6).alias("mean"),
    spark_round(stddev("ride_length_minutes"),  6).alias("std"),
    min("ride_length_minutes").alias("min"),
    percentile_approx("ride_length_minutes", 0.25).alias("25%"),
    percentile_approx("ride_length_minutes", 0.50).alias("50%"),
    percentile_approx("ride_length_minutes", 0.75).alias("75%"),
    max("ride_length_minutes").alias("max"),
).show()

# Trip share by user type
total = df_clean.count()
print("Trip share by user type:")
(
    df_clean.groupBy("member_casual")
    .count()
    .withColumn("share_%", spark_round(col("count") / total * 100, 2))
    .orderBy(col("count").desc())
    .show()
)

print(f"\nCell time: {time.time() - _t:.2f}s")

Ride length (minutes) by user type:
+-------------+-------+---------+---------+--------------------+-----------------+------------------+------------------+------------------+
|member_casual|  count|     mean|      std|                 min|              25%|               50%|               75%|               max|
+-------------+-------+---------+---------+--------------------+-----------------+------------------+------------------+------------------+
|       casual|6202648|27.213075|49.473105|0.016666666666666666|8.733333333333333|15.733333333333333|29.366666666666667|1439.9333333333334|
|       member|8577218|13.441199|20.328097|0.016666666666666666|             5.55| 9.633333333333333|16.766666666666666|           1439.95|
+-------------+-------+---------+---------+--------------------+-----------------+------------------+------------------+------------------+

Trip share by user type:
+-------------+-------+-------+
|member_casual|  count|share_%|
+-------------+-------+-------+
| 

## Time and day patterns

Examine how rides vary by hour of day and day of week for members vs casual riders.

In [9]:
_t = time.time()

df_clean = (
    df_clean
    .withColumn("hour",        hour("started_at"))
    .withColumn("day_of_week", date_format("started_at", "EEEE"))
)

df_clean.select("started_at", "hour", "day_of_week").show(5, truncate=False)

print(f"\nCell time: {time.time() - _t:.2f}s")

+-------------------+----+-----------+
|started_at         |hour|day_of_week|
+-------------------+----+-----------+
|2020-04-26 17:45:14|17  |Sunday     |
|2020-04-17 17:08:54|17  |Friday     |
|2020-04-01 17:54:13|17  |Wednesday  |
|2020-04-07 12:50:19|12  |Tuesday    |
|2020-04-18 10:22:59|10  |Saturday   |
+-------------------+----+-----------+
only showing top 5 rows


Cell time: 0.45s


In [11]:
_t = time.time()

print("Trips by hour of day and user type:")
(
    df_clean
    .groupBy("hour", "member_casual")
    .count()
    .orderBy("hour", "member_casual")
    .show(48)
)

print(f"\nCell time: {time.time() - _t:.2f}s")

Trips by hour of day and user type:
+----+-------------+------+
|hour|member_casual| count|
+----+-------------+------+
|   0|       casual|121180|
|   0|       member| 81058|
|   1|       casual| 82038|
|   1|       member| 50555|
|   2|       casual| 51536|
|   2|       member| 29161|
|   3|       casual| 28939|
|   3|       member| 17541|
|   4|       casual| 20612|
|   4|       member| 21105|
|   5|       casual| 29999|
|   5|       member| 81910|
|   6|       casual| 67403|
|   6|       member|236404|
|   7|       casual|120264|
|   7|       member|434907|
|   8|       casual|163761|
|   8|       member|507716|
|   9|       casual|187663|
|   9|       member|369770|
|  10|       casual|253807|
|  10|       member|358683|
|  11|       casual|333910|
|  11|       member|434457|
|  12|       casual|399206|
|  12|       member|504251|
|  13|       casual|424936|
|  13|       member|498956|
|  14|       casual|445917|
|  14|       member|494279|
|  15|       casual|479892|
|  15|      

In [12]:
_t = time.time()

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Build a sort-index map so days appear Mon–Sun rather than alphabetically
day_index = F.create_map(
    *[v for day in DAY_ORDER for v in (F.lit(day), F.lit(DAY_ORDER.index(day)))]
)

print("Trips by day of week and user type:")
(
    df_clean
    .groupBy("day_of_week", "member_casual")
    .count()
    .withColumn("_sort", day_index[col("day_of_week")])
    .orderBy("_sort", "member_casual")
    .drop("_sort")
    .show(14)
)

print(f"\nCell time: {time.time() - _t:.2f}s")

Trips by day of week and user type:
+-----------+-------------+-------+
|day_of_week|member_casual|  count|
+-----------+-------------+-------+
|     Monday|       casual| 704022|
|     Monday|       member|1180546|
|    Tuesday|       casual| 674169|
|    Tuesday|       member|1298043|
|  Wednesday|       casual| 704321|
|  Wednesday|       member|1329205|
|   Thursday|       casual| 756411|
|   Thursday|       member|1311182|
|     Friday|       casual| 898709|
|     Friday|       member|1237934|
|   Saturday|       casual|1341744|
|   Saturday|       member|1187199|
|     Sunday|       casual|1123272|
|     Sunday|       member|1033109|
+-----------+-------------+-------+


Cell time: 2.07s


## Station-based insights

In [13]:
_t = time.time()

for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    (
        df_clean
        .filter((col("member_casual") == user_type) & col("start_station_name").isNotNull())
        .groupBy("start_station_name")
        .count()
        .orderBy(col("count").desc())
        .show(10, truncate=False)
    )

    print(f"\nTop 10 end stations for {user_type}s:")
    (
        df_clean
        .filter((col("member_casual") == user_type) & col("end_station_name").isNotNull())
        .groupBy("end_station_name")
        .count()
        .orderBy(col("count").desc())
        .show(10, truncate=False)
    )

print(f"\nCell time: {time.time() - _t:.2f}s")


Top 10 start stations for members:
+----------------------------+-----+
|start_station_name          |count|
+----------------------------+-----+
|Clark St & Elm St           |67158|
|Kingsbury St & Kinzie St    |65084|
|Wells St & Concord Ln       |60483|
|Wells St & Elm St           |53852|
|Dearborn St & Erie St       |51996|
|Broadway & Barry Ave        |51286|
|St. Clair St & Erie St      |51274|
|Wells St & Huron St         |50316|
|Clinton St & Madison St     |50193|
|Clinton St & Washington Blvd|46653|
+----------------------------+-----+
only showing top 10 rows


Top 10 end stations for members:
+----------------------------+-----+
|end_station_name            |count|
+----------------------------+-----+
|Clark St & Elm St           |68326|
|Kingsbury St & Kinzie St    |65008|
|Wells St & Concord Ln       |62020|
|Dearborn St & Erie St       |53395|
|Wells St & Elm St           |53381|
|St. Clair St & Erie St      |53149|
|Broadway & Barry Ave        |52777|
|Clinton St & Ma

In [15]:
_t = time.time()

print("Average ride length (minutes) by start station and user type (>= 30 trips):")
(
    df_clean
    .filter(col("start_station_name").isNotNull())
    .groupBy("start_station_name", "member_casual")
    .agg(
        count("ride_id").alias("count"),
        spark_round(avg("ride_length_minutes"), 2).alias("mean"),
    )
    .filter(col("count") >= 30)
    .orderBy(col("mean").desc())
    .show(20, truncate=False)
)

print(f"\nCell time: {time.time() - _t:.2f}s")

Average ride length (minutes) by start station and user type (>= 30 trips):
+--------------------------------+-------------+-----+------+
|start_station_name              |member_casual|count|mean  |
+--------------------------------+-------------+-----+------+
|Ashland Ave & 69th St           |casual       |44   |102.83|
|Racine Ave & 65th St            |casual       |165  |88.12 |
|Altgeld Gardens                 |casual       |158  |76.48 |
|Yates Blvd & 75th St            |casual       |526  |72.45 |
|Ashland Ave & 63rd St           |casual       |331  |68.77 |
|Halsted St & 59th St            |casual       |152  |66.97 |
|Jeffery Blvd & 76th St          |casual       |490  |64.13 |
|Commercial Ave & 100th St       |casual       |125  |62.64 |
|Lake Park Ave & 35th St         |casual       |6368 |59.52 |
|Phillips Ave & 79th St          |casual       |653  |58.71 |
|Central Park Blvd & 5th Ave     |casual       |502  |58.1  |
|Rainbow Beach                   |casual       |672  |58

## Segment comparisons for business insights

Weekend vs weekday, commute vs leisure, ride length bands.

In [16]:
_t = time.time()

COMMUTE_HOURS = list(range(6, 10)) + list(range(16, 20))

df_clean = (
    df_clean
    .withColumn("is_weekend",      col("day_of_week").isin(["Saturday", "Sunday"]))
    .withColumn("is_commute_hour", col("hour").isin(COMMUTE_HOURS))
    .withColumn(
        "duration_band",
        when(col("ride_length_minutes") <= 15, "short (<=15m)")
        .when(col("ride_length_minutes") <= 45, "medium (15-45m)")
        .otherwise("long (>45m)"),
    )
)

# Weekend vs weekday
print("Weekend vs weekday trips by user type:")
(
    df_clean
    .groupBy("member_casual", "is_weekend")
    .count()
    .orderBy("member_casual", "is_weekend")
    .show()
)

# Commute vs non-commute hours
print("Commute-hour vs other-hour trips by user type:")
(
    df_clean
    .groupBy("member_casual", "is_commute_hour")
    .count()
    .orderBy("member_casual", "is_commute_hour")
    .show()
)

# Ride length band distribution (% of trips per user type)
band_counts = df_clean.groupBy("member_casual", "duration_band").count()
totals      = (
    df_clean.groupBy("member_casual")
    .count()
    .withColumnRenamed("count", "total")
)

print("Ride length bands (% of trips) by user type:")
(
    band_counts
    .join(totals, "member_casual")
    .withColumn("pct_%", spark_round(col("count") / col("total") * 100, 1))
    .select("member_casual", "duration_band", "pct_%")
    .orderBy("member_casual", "duration_band")
    .show()
)

print(f"\nCell time: {time.time() - _t:.2f}s")

Weekend vs weekday trips by user type:
+-------------+----------+-------+
|member_casual|is_weekend|  count|
+-------------+----------+-------+
|       casual|     false|3737632|
|       casual|      true|2465016|
|       member|     false|6356910|
|       member|      true|2220308|
+-------------+----------+-------+

Commute-hour vs other-hour trips by user type:
+-------------+---------------+-------+
|member_casual|is_commute_hour|  count|
+-------------+---------------+-------+
|       casual|          false|3611959|
|       casual|           true|2590689|
|       member|          false|4093898|
|       member|           true|4483320|
+-------------+---------------+-------+

Ride length bands (% of trips) by user type:
+-------------+---------------+-----+
|member_casual|  duration_band|pct_%|
+-------------+---------------+-----+
|       casual|    long (>45m)| 14.1|
|       casual|medium (15-45m)| 38.0|
|       casual|  short (<=15m)| 47.9|
|       member|    long (>45m)|  1.8|
|

## Total run time

In [17]:
_total = time.time() - _NB_START
_mins  = int(_total // 60)
_secs  = _total % 60
print(f"Total notebook time: {_mins}m {_secs:.2f}s  ({_total:.2f}s)")

Total notebook time: 2m 48.44s  (168.44s)
